In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Import your newly modified modules
# (Adjust imports to match your folder structure)
from environment.environment import GraphWorldMFG_MultiGroup
from solver.solver import GraphMFG_OMD_EdgeSolver_MultiGroup, solve_multigroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer

# Ensure deterministic outputs for verification
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running tests on device: {device}")

# ==========================================
# STEP 1: Define a Toy Graph with a Bottleneck
# ==========================================
# We create a 5-node graph:
# Node 0 (Source) -> Node 1 & Node 2 (Alternative paths)
# Node 1 & 2 -> Node 3 (Bottleneck junction) -> Node 4 (Sink)
adj_matrix = np.zeros((4, 4))
adj_matrix[0, 1] = 1.0  # Path A
adj_matrix[0, 2] = 1.0  # Path B
adj_matrix[1, 2] = 1.0  # Path B
adj_matrix[2, 1] = 1.0  # Path B
adj_matrix[1, 3] = 1.0  # Merging into bottleneck
adj_matrix[2, 3] = 1.0

adj_matrix += np.eye((4))

# Group definition: Mass travels from 0 to 4
groups = [
    {"source": 0, "sink": 4, "mass": 1.0} # High mass to force congestion!
]

env = GraphWorldMFG_MultiGroup(adj_matrix, groups, device=device)

# ==========================================
# STEP 2: Initialize Solvers and Trainer
# ==========================================
H = 15       # Physical time horizon
W_max = 4    # Max congestion wait cycles

solvers = [
    GraphMFG_OMD_EdgeSolver_MultiGroup(env, group_idx=0, eta=0.1, tau=0.01, H=H)
]

trainer = GraphEdgeMFG_Trainer(env, solvers)

# ==========================================
# STEP 3: Execute the Waiting-Time MFG
# ==========================================
print("\n--- Testing solve_multigroup loop ---")
# Baseline: Zero incentives from the leader (pure user equilibrium search)
theta_list = [torch.zeros((env.N, env.N), device=device) for _ in range(env.K)]

flows, final_flows, policies = solve_multigroup(
    env, solvers, theta_list, T=450, W_max=W_max
)

print(f"Flow tensor output shape: {flows.shape}")       # Expected: (1, 15, 5, 5)
print(f"Spatial combined flow shape: {final_flows.shape}") # Expected: (15, 5)
print("Policy optimization execution complete!")

# ==========================================
# STEP 4: Assertions and Reality-Checks
# ==========================================
print("\n--- Verifying Simulation Rules ---")

# Check 1: Mass Conservation
# Total mass in the system across all nodes and wait states must equal the starting mass (10.0)
for h in range(H):
    total_mass_at_h = flows[0, h].sum().item()
    print(f"Timestep h={h:2d} | Total mass conserved: {total_mass_at_h:.2f}")
    assert np.isclose(total_mass_at_h, 1.0), f"Mass leak detected at step {h}!"

# Check 2: Sink Absorption Mechanics
# At the final time horizon step, some population mass should have gathered at the sink (Node 4) at w=0
final_sink_mass = flows[0, -1, 3, 0].item()
print(f"\nFinal population reaching the destination sink at h={H-1}: {final_sink_mass:.2f} / 1.0")

# Check 3: Latency & Waiting-Time Check
# Look at the bottleneck node 3 heading to 4. Because mass is high, agents should be trapped in w > 0 states.
trapped_mass = flows[0, :, 3, 1:].sum().item()
print(f"Total historical mass detected waiting inside congestion queues (w > 0): {trapped_mass:.2f}")

# Check 4: Trainer Integration Check
print("\n--- Testing Single Trainer Step ---")
loss, updated_flows = trainer.train_step(flows, torch.zeros((H, env.N, env.N), device=device))
print(f"Trainer social loss calculated: {loss:.2f}")
print("Test Pipeline Successful!")

Running tests on device: cpu

--- Testing solve_multigroup loop ---
Flow tensor output shape: torch.Size([1, 15, 4, 5])
Spatial combined flow shape: torch.Size([15, 4])
Policy optimization execution complete!

--- Verifying Simulation Rules ---
Timestep h= 0 | Total mass conserved: 1.00
Timestep h= 1 | Total mass conserved: 1.00
Timestep h= 2 | Total mass conserved: 1.00
Timestep h= 3 | Total mass conserved: 1.00
Timestep h= 4 | Total mass conserved: 1.00
Timestep h= 5 | Total mass conserved: 1.00
Timestep h= 6 | Total mass conserved: 1.00
Timestep h= 7 | Total mass conserved: 1.00
Timestep h= 8 | Total mass conserved: 1.00
Timestep h= 9 | Total mass conserved: 1.00
Timestep h=10 | Total mass conserved: 1.00
Timestep h=11 | Total mass conserved: 1.00
Timestep h=12 | Total mass conserved: 1.00
Timestep h=13 | Total mass conserved: 1.00
Timestep h=14 | Total mass conserved: 1.00

Final population reaching the destination sink at h=14: 0.99 / 1.0
Total historical mass detected waiting ins

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x272 and 186x32)

In [5]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.0596e-01, 1.8808e-01, 4.0596e-01, 0.0000e+00],
        [1.3532e-01, 3.3333e-01, 3.3333e-01, 1.9801e-01],
        [4.5107e-02, 2.6733e-01, 2.6733e-01, 4.2024e-01],
        [1.5036e-02, 1.9325e-01, 1.9325e-01, 5.9845e-01],
        [5.0119e-03, 1.3385e-01, 1.3385e-01, 7.2729e-01],
        [1.6706e-03, 9.0903e-02, 9.0903e-02, 8.1652e-01],
        [5.5687e-04, 6.1159e-02, 6.1159e-02, 8.7713e-01],
        [1.8562e-04, 4.0958e-02, 4.0958e-02, 9.1790e-01],
        [6.1875e-05, 2.7367e-02, 2.7367e-02, 9.4520e-01],
        [2.0625e-05, 1.8266e-02, 1.8266e-02, 9.6345e-01],
        [6.8750e-06, 1.2184e-02, 1.2184e-02, 9.7563e-01],
        [2.2917e-06, 8.1249e-03, 8.1249e-03, 9.8375e-01],
        [7.6389e-07, 5.4174e-03, 5.4174e-03, 9.8916e-01],
        [2.5463e-07, 3.6118e-03, 3.6118e-03, 9.9278e-01]])